# D.3 · SFT QLoRA trên Kaggle
Mục tiêu pilot: từ **release D.2 đã duyệt → adapter → checkpoint nối lại được**. Không dùng 24 QA khi còn ở trạng thái ứng viên, không dùng phiếu kiểm hoặc khảo sát/test. Notebook mặc định không upload, tải mô hình, cài gói hay huấn luyện. Đừng gắn dữ liệu khảo sát vào notebook này.

1. Trên máy nhóm: chạy `python scripts/package_d3_kaggle.py`. Import notebook này vào **Notebook Private**; gắn gói code qua **Add Input** (Dataset Private). Chưa có release thì dừng ở kiểm CLI trên CPU.
2. Sau khi D.2 phát hành và nhóm cho phép dùng đúng nền tảng: gắn riêng gói dữ liệu tối thiểu, điền đường dẫn và SHA-256 manifest giữ ở máy nhóm. Dùng CPU để kiểm release trước. Chỉ kích hoạt một GPU khi thật sự bắt đầu SFT.
3. Khi chạy GPU lần đầu, cần Internet cho model/tokenizer công khai nếu chưa có trong cache đúng revision. Không gửi QA cho API; notebook huấn luyện trong máy ảo Kaggle. Giữ cả output lẫn version ở chế độ riêng tư.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile
from importlib import metadata

# Trỏ tới file ZIP HOẶC thư mục mà Kaggle đã giải nén (có scripts/ hoặc manifest.json ngay bên dưới).
CODE_INPUT = None
RELEASE_INPUT = None
MANIFEST_SHA256 = ''  # Hash của manifest D.2 đã đối chiếu trên máy nhóm, không phải hash ZIP.
INSTALL_PACKAGES = False
RUN_TRAIN = False
RANK, ALPHA, LR, EPOCHS, SEED, MAX_LENGTH = 8, 16, 1e-4, 2, 41, 512
OUTPUT = Path('/kaggle/working') / f'd3-r{RANK}-a{ALPHA}-lr{LR}-e{EPOCHS}-s{SEED}'
RESUME = None  # Checkpoint của chính nhóm từ một version Kaggle Private đã lưu; dùng OUTPUT mới.

### 1 · Nạp đúng các tệp đã chỉ định
Khi chạy trong bản clone, notebook có thể tìm script ở thư mục gốc. Trên Kaggle, dùng **Copy path** của input. Không quét các dataset khác. Không giải nén toàn bộ archive vào thư mục làm việc.

In [ ]:
release_ok = False
scratch = tempfile.TemporaryDirectory(prefix='d3-input-')
code_files = ('scripts/train_d3_sft.py', 'src/d3_data.py', 'requirements-d3.txt')
release_files = ('manifest.json', 'train.jsonl', 'validation.jsonl')

def open_input(path, required):
    p = Path(path)
    if p.is_dir():
        if not all((p / name).is_file() for name in required):
            raise ValueError(f'Thiếu tệp bắt buộc trong {p}; kiểm lại cấp thư mục.')
        return p.resolve()
    dest = Path(tempfile.mkdtemp(dir=scratch.name))
    with zipfile.ZipFile(p) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(required):
            raise ValueError('ZIP không đúng danh sách tệp của bộ đóng gói D.3.')
        for name in required:
            target = dest / name
            target.parent.mkdir(exist_ok=True, parents=True)
            with archive.open(name) as src, target.open('xb') as out:
                shutil.copyfileobj(src, out)
    return dest

local = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / code_files[0]).is_file()), None)
code = open_input(CODE_INPUT or local, code_files) if CODE_INPUT or local else None
release = open_input(RELEASE_INPUT, release_files) if RELEASE_INPUT else None
if code:
    cli = [sys.executable, str(code / code_files[0])]
    subprocess.run(cli + ['--help'], check=True)
else:
    print('Chưa gắn code. Chọn Add Input cho gói d3-code rồi điền CODE_INPUT.')

if release:
    if not code or hashlib.sha256((release / 'manifest.json').read_bytes()).hexdigest() != MANIFEST_SHA256:
        raise ValueError('Thiếu code hoặc sai hash manifest so với release được phép.')
    args = ['--release', str(release), '--output', str(OUTPUT), '--seed', str(SEED), '--rank', str(RANK),
            '--alpha', str(ALPHA), '--lr', str(LR), '--epochs', str(EPOCHS), '--max-length', str(MAX_LENGTH)]
    subprocess.run(cli + args + ['--check-only'], check=True)
    release_ok = True
else:
    print('Chưa gắn release D.2: không có QA nào được đọc, không chạy SFT.')

### 2 · Môi trường GPU (chỉ sau khi release qua cổng)
Nếu lần đầu báo thiếu/sai phiên bản, xem danh sách rồi bật `INSTALL_PACKAGES` khi được phép tải gói. Ô chỉ cài khi kế hoạch của pip không thay PyTorch, Triton hoặc thư viện NVIDIA. Không tháo/cài lại CUDA để vượt cổng. Script huấn luyện chạy ở tiến trình mới.

In [ ]:
if INSTALL_PACKAGES or RUN_TRAIN:
    if not release_ok:
        raise ValueError('Phải kiểm release ở ô 1 trước.')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Chọn Session options → Accelerator → GPU; không chạy SFT trên CPU.')
    print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__, 'CUDA:', torch.version.cuda)
    pins = (code / 'requirements-d3.txt').read_text().splitlines()
    pins = [s for s in pins if s and not s.startswith('#')]
    def installed(pkg):
        try:
            return metadata.version(pkg)
        except metadata.PackageNotFoundError:
            return None
    missing = [pin for pin in pins if installed(pin.split('==')[0]) != pin.split('==')[1]]
    if missing and not INSTALL_PACKAGES:
        raise RuntimeError(f'Gói cần khớp: {missing}. Xem hướng dẫn trước khi bật INSTALL_PACKAGES.')
    if missing:
        constraint = Path(scratch.name) / 'torch-pin.txt'
        constraint.write_text('torch==' + metadata.version('torch'))
        report = Path(scratch.name) / 'pip-plan.json'
        pip = [sys.executable, '-m', 'pip', 'install', '-c', str(constraint), *pins]
        subprocess.run(pip + ['--dry-run', '--report', str(report)], check=True)
        changes = [x['metadata']['name'].lower() for x in json.loads(report.read_text())['install']]
        if any(n.startswith(('torch', 'triton', 'nvidia', 'cuda')) for n in changes):
            raise RuntimeError(f'Kế hoạch thay stack GPU: {changes}. Dừng, không cài.')
        subprocess.run(pip, check=True)
else:
    print('Không thay đổi môi trường.')

### 3 · Chạy một cấu hình
Bật `RUN_TRAIN` sau khi cổng release đạt. Mỗi cấu hình/seed dùng thư mục mới. **Save Version → Save & Run All** hoặc lưu toàn bộ output riêng trước khi mất phiên. Để chứng minh resume: giữ checkpoint epoch 1 của run 2 epoch, mở phiên sạch, gắn version đã lưu qua **Add Input**, trỏ `RESUME` tới checkpoint đó, đổi `OUTPUT` sang thư mục mới và giữ nguyên toàn bộ tham số. So sánh epoch/số bước và NLL cuối với run liên tục, ghi sai khác thực tế. Adapter dùng suy luận không thay checkpoint resume.

In [ ]:
if RUN_TRAIN:
    if not release_ok:
        raise ValueError('Chưa xác minh release ở ô 1.')
    env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY='1', WANDB_DISABLED='true', CUDA_VISIBLE_DEVICES='0')
    command = cli + args + (['--resume', str(RESUME)] if RESUME else [])
    subprocess.run(command, env=env, check=True)
else:
    print('RUN_TRAIN=False: chưa tải trọng số hay huấn luyện.')